# Lemma Fine-tuning — ByT5-base
**Task:** seq2seq — given a word form in context, generate its lemma.  
**Model:** `google/byt5-base` — byte-level T5, no vocabulary: handles OOV dialect forms perfectly.  
**Data:** `tor_train/dev/test.tsv` — `form\tlemma\txpos`, blank-line sentence boundaries.  
**Runtime:** L4 GPU (24 GB) recommended.

Upload the `dataset/` folder to `MyDrive/TorlakTag/dataset/` before running.  

**Input format:** `lemmatize: {left_3_words} [X] {form} [/X] {right_3_words}`  
**Output:** `{lemma}`

In [ ]:
!pip install transformers torch sentencepiece -q

In [ ]:
import os, random
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIG ────────────────────────────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/TorlakTag'
TRAIN_TSV   = f'{DRIVE_ROOT}/dataset/tor_train.tsv'
DEV_TSV     = f'{DRIVE_ROOT}/dataset/tor_dev.tsv'
TEST_TSV    = f'{DRIVE_ROOT}/dataset/tor_test.tsv'
SAVE_DIR    = f'{DRIVE_ROOT}/models/byt5_lemma'

MODEL_NAME      = 'google/byt5-base'
CONTEXT_WINDOW  = 3      # words on each side as context
MAX_SRC_LEN     = 128    # bytes — generous for ~10 context words
MAX_TGT_LEN     = 64     # bytes — lemmas are short
BATCH_SIZE      = 32
LR              = 5e-4   # T5 models use higher LR than encoder-only
WEIGHT_DECAY    = 0.01
EPOCHS          = 10
WARMUP_RATIO    = 0.1
EVAL_SAMPLES    = 500    # dev samples evaluated per epoch (full eval only at end)
SEED            = 42

os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def load_tsv(path):
    """Parse form\tlemma\txpos TSV; blank lines = sentence boundaries.
    Handles both truly-empty lines and tab-only boundary rows (\t\t).
    """
    sentences, current = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():           # blank OR tab-only boundary lines
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split('\t')
                if len(parts) == 3 and parts[0].strip():
                    current.append({'form': parts[0], 'lemma': parts[1], 'xpos': parts[2]})
    if current:
        sentences.append(current)
    return sentences

train_sents = load_tsv(TRAIN_TSV)
dev_sents   = load_tsv(DEV_TSV)
test_sents  = load_tsv(TEST_TSV)

for split, sents in [('train', train_sents), ('dev', dev_sents), ('test', test_sents)]:
    toks = sum(len(s) for s in sents)
    print(f'{split:5s}: {len(sents):5d} sentences  {toks:6d} tokens')

In [ ]:
def make_pairs(sentences, ctx=CONTEXT_WINDOW):
    """
    Build (source, target) pairs for every token.

    source = 'lemmatize: {left} [X] {form} [/X] {right}'
    target = lemma

    ByT5 reads every byte, so the [X] / [/X] markers are just ASCII strings —
    no special token IDs needed.
    """
    pairs = []
    for sent in sentences:
        for i, tok in enumerate(sent):
            left  = ' '.join(t['form'] for t in sent[max(0, i - ctx): i])
            right = ' '.join(t['form'] for t in sent[i + 1: i + 1 + ctx])
            src   = f"lemmatize: {left} [X] {tok['form']} [/X] {right}".strip()
            pairs.append({'source': src, 'target': tok['lemma']})
    return pairs

train_pairs = make_pairs(train_sents)
dev_pairs   = make_pairs(dev_sents)
test_pairs  = make_pairs(test_sents)

print(f'Pairs — train: {len(train_pairs)}  dev: {len(dev_pairs)}  test: {len(test_pairs)}')
print('\nSample pairs:')
for p in train_pairs[1:4]:
    print(f'  SRC: {p["source"]}')
    print(f'  TGT: {p["target"]}')
    print()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(pairs, max_src=MAX_SRC_LEN, max_tgt=MAX_TGT_LEN):
    sources = [p['source'] for p in pairs]
    targets = [p['target'] for p in pairs]

    # Tokenize source (bytes)
    enc = tokenizer(
        sources,
        max_length=max_src, truncation=True, padding='max_length',
        return_tensors='pt',
    )

    # Tokenize target (bytes) — use text_target for seq2seq
    with tokenizer.as_target_tokenizer():
        lbl_enc = tokenizer(
            targets,
            max_length=max_tgt, truncation=True, padding='max_length',
            return_tensors='pt',
        )
    labels = lbl_enc['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100   # ignore padding in loss

    enc['labels'] = labels
    return enc

print('Encoding train …')
train_enc = encode(train_pairs)
print('Encoding dev …')
dev_enc   = encode(dev_pairs)
print('Encoding test …')
test_enc  = encode(test_pairs)
print(f'Train input_ids shape: {train_enc["input_ids"].shape}')

In [ ]:
from torch.utils.data import Dataset, DataLoader

class LemmaDataset(Dataset):
    def __init__(self, enc):
        self.ids   = enc['input_ids']
        self.mask  = enc['attention_mask']
        self.lbls  = enc['labels']
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return {'input_ids': self.ids[i],
                'attention_mask': self.mask[i],
                'labels': self.lbls[i]}

# num_workers=0: Colab multiprocessing causes semaphore/fd crashes with workers > 0
train_loader = DataLoader(LemmaDataset(train_enc), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
dev_loader   = DataLoader(LemmaDataset(dev_enc),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(LemmaDataset(test_enc),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
print(f'Batches — train: {len(train_loader)}  dev: {len(dev_loader)}  test: {len(test_loader)}')

In [ ]:
from transformers import AutoModelForSeq2SeqLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

PAD = tokenizer.pad_token_id

def evaluate(loader, max_samples=None):
    """
    Exact-match accuracy.  max_samples limits evaluation for speed during training.
    """
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            if max_samples and total >= max_samples:
                break
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].clone()
            lbls[lbls == -100] = PAD    # restore pad so we can decode

            gen = model.generate(
                ids, attention_mask=mask,
                max_new_tokens=MAX_TGT_LEN,
                num_beams=4,
            )
            for g, l in zip(gen, lbls):
                pred = tokenizer.decode(g, skip_special_tokens=True).strip()
                gold = tokenizer.decode(l[l != PAD], skip_special_tokens=True).strip()
                correct += int(pred == gold)
                total   += 1
    return correct / total if total else 0.0

best_dev = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        running += loss.item()

    # Fast dev eval on a subset during training; full eval at end
    dev_acc = evaluate(dev_loader, max_samples=EVAL_SAMPLES)
    print(f'Epoch {epoch}  loss={running/len(train_loader):.4f}  '
          f'dev_acc(n={EVAL_SAMPLES})={dev_acc:.4f}')

    if dev_acc > best_dev:
        best_dev = dev_acc
        model.save_pretrained(f'{SAVE_DIR}/best')
        tokenizer.save_pretrained(f'{SAVE_DIR}/best')
        print(f'  ✓ Saved  (best dev so far: {best_dev:.4f})')

In [ ]:
import Levenshtein   # pip install python-Levenshtein

# Reload best checkpoint
model = AutoModelForSeq2SeqLM.from_pretrained(f'{SAVE_DIR}/best').to(device)

# Full test-set evaluation
test_acc = evaluate(test_loader)
print(f'Test exact-match accuracy: {test_acc:.4f}')

# Collect errors for analysis
model.eval()
errors = []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].clone()
        lbls[lbls == -100] = PAD
        gen  = model.generate(ids, attention_mask=mask,
                               max_new_tokens=MAX_TGT_LEN, num_beams=4)
        for g, l, src in zip(gen, lbls, batch['input_ids']):
            pred  = tokenizer.decode(g,           skip_special_tokens=True).strip()
            gold  = tokenizer.decode(l[l != PAD], skip_special_tokens=True).strip()
            inp   = tokenizer.decode(src,         skip_special_tokens=True).strip()
            if pred != gold:
                edit = Levenshtein.distance(pred, gold)
                errors.append({'input': inp, 'pred': pred, 'gold': gold, 'edit': edit})

avg_edit = sum(e['edit'] for e in errors) / len(errors) if errors else 0
print(f'Errors: {len(errors)} / {len(test_pairs)}  avg edit-distance: {avg_edit:.2f}')

print('\nFirst 20 errors:')
print(f'{"input":<45} {"pred":<20} {"gold":<20} edit')
print('-' * 95)
for e in sorted(errors, key=lambda x: -x['edit'])[:20]:
    print(f"{e['input']:<45} {e['pred']:<20} {e['gold']:<20} {e['edit']}")

In [ ]:
def lemmatize_sentence(sentence_words):
    """Lemmatize every token in a list of word forms."""
    model.eval()
    results = []
    for i, form in enumerate(sentence_words):
        left  = ' '.join(sentence_words[max(0, i - CONTEXT_WINDOW): i])
        right = ' '.join(sentence_words[i + 1: i + 1 + CONTEXT_WINDOW])
        src   = f'lemmatize: {left} [X] {form} [/X] {right}'.strip()

        enc = tokenizer(src, return_tensors='pt',
                        max_length=MAX_SRC_LEN, truncation=True).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_TGT_LEN, num_beams=4)
        lemma = tokenizer.decode(gen[0], skip_special_tokens=True).strip()
        results.append((form, lemma))
    return results

# Example from the Torlak transcript
example = ['i', 'on', 'pade', 'i', 'tolko', 'se', 'ubije', 'mnogo']
print(f'{"form":<18} lemma')
for form, lemma in lemmatize_sentence(example):
    print(f'{form:<18} {lemma}')